# dentify bias in the model predictions by implementing fairness metrics such as disparate impact and equalized odds without using prebuilt fairness libraries, manually computing bias scores across demographic groups.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

In [2]:
df = pd.read_csv("heart.csv")

print(df.head())
print(df.columns)

target_col = "HeartDisease"
sensitive_col = "Sex"

df = df.dropna(subset=[target_col, sensitive_col])

y = df[target_col].astype(int)
sensitive = df[sensitive_col].astype(str)

# Sensitive attribute model માંથી exclude કરીએ છીએ
X = df.drop(columns=[target_col, sensitive_col])

   Age Sex ChestPainType  RestingBP  Cholesterol  FastingBS RestingECG  MaxHR  \
0   40   M           ATA        140          289          0     Normal    172   
1   49   F           NAP        160          180          0     Normal    156   
2   37   M           ATA        130          283          0         ST     98   
3   48   F           ASY        138          214          0     Normal    108   
4   54   M           NAP        150          195          0     Normal    122   

  ExerciseAngina  Oldpeak ST_Slope  HeartDisease  
0              N      0.0       Up             0  
1              N      1.0     Flat             1  
2              N      0.0       Up             0  
3              Y      1.5     Flat             1  
4              N      0.0       Up             0  
Index(['Age', 'Sex', 'ChestPainType', 'RestingBP', 'Cholesterol', 'FastingBS',
       'RestingECG', 'MaxHR', 'ExerciseAngina', 'Oldpeak', 'ST_Slope',
       'HeartDisease'],
      dtype='str')


In [3]:
X_train, X_test, y_train, y_test, s_train, s_test = train_test_split(
    X, y, sensitive,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [4]:
num_cols = X_train.select_dtypes(include=np.number).columns
cat_cols = X_train.select_dtypes(exclude=np.number).columns

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

In [5]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced"
    ))
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

Accuracy: 0.875
F1 Score: 0.8888888888888888


In [6]:
def calculate_metrics(y_true, y_pred, groups):

    data = pd.DataFrame({
        "Actual": np.array(y_true),
        "Predicted": np.array(y_pred),
        "Group": np.array(groups)
    })

    results = []

    for group in data["Group"].unique():

        group_data = data[data["Group"] == group]

        actual = group_data["Actual"]
        predicted = group_data["Predicted"]

        TP = ((actual == 1) & (predicted == 1)).sum()
        TN = ((actual == 0) & (predicted == 0)).sum()
        FP = ((actual == 0) & (predicted == 1)).sum()
        FN = ((actual == 1) & (predicted == 0)).sum()

        selection_rate = (TP + FP) / len(group_data)

        TPR = TP / (TP + FN) if (TP + FN) > 0 else np.nan

        FPR = FP / (FP + TN) if (FP + TN) > 0 else np.nan

        results.append({
            "Group": group,
            "Total": len(group_data),
            "TP": TP,
            "TN": TN,
            "FP": FP,
            "FN": FN,
            "Selection Rate": selection_rate,
            "TPR": TPR,
            "FPR": FPR
        })

    return pd.DataFrame(results)


fairness_report = calculate_metrics(
    y_test, y_pred, s_test
)

print(fairness_report)

  Group  Total  TP  TN  FP  FN  Selection Rate       TPR      FPR
0     M    146  86  40  10  10        0.657534  0.895833  0.20000
1     F     38   6  29   3   0        0.236842  1.000000  0.09375


In [7]:
rates = fairness_report["Selection Rate"].dropna()

if len(rates) >= 2 and rates.max() > 0:
    DI = rates.min() / rates.max()
else:
    DI = np.nan

print("Disparate Impact:", DI)

Disparate Impact: 0.36019736842105265


In [8]:
tpr_values = fairness_report["TPR"].dropna()
fpr_values = fairness_report["FPR"].dropna()

TPR_Gap = (
    tpr_values.max() - tpr_values.min()
    if len(tpr_values) >= 2 else np.nan
)

FPR_Gap = (
    fpr_values.max() - fpr_values.min()
    if len(fpr_values) >= 2 else np.nan
)

Equalized_Odds_Gap = (
    max(TPR_Gap, FPR_Gap)
    if pd.notna(TPR_Gap) and pd.notna(FPR_Gap)
    else np.nan
)

print("TPR Gap:", TPR_Gap)
print("FPR Gap:", FPR_Gap)
print("Equalized Odds Gap:", Equalized_Odds_Gap)

TPR Gap: 0.10416666666666663
FPR Gap: 0.10625000000000001
Equalized Odds Gap: 0.10625000000000001


In [9]:
print("========== FAIRNESS REPORT ==========")

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))

print("F1 Score:", round(f1_score(y_test, y_pred), 4))

print("Disparate Impact:", round(DI, 4) if pd.notna(DI) else "N/A")

print("TPR Gap:", round(TPR_Gap, 4) if pd.notna(TPR_Gap) else "N/A")

print("FPR Gap:", round(FPR_Gap, 4) if pd.notna(FPR_Gap) else "N/A")

print("Equalized Odds Gap:",
      round(Equalized_Odds_Gap, 4)
      if pd.notna(Equalized_Odds_Gap) else "N/A")

fairness_report.to_csv("fairness_report.csv", index=False)

========== FAIRNESS REPORT ==========
Accuracy: 0.875
F1 Score: 0.8889
Disparate Impact: 0.3602
TPR Gap: 0.1042
FPR Gap: 0.1062
Equalized Odds Gap: 0.1062
